# 05 — Task Decomposition

**CCA Pattern**: The coordinator decomposes queries into parallel and sequential phases based on data dependencies.

Web search and document analysis can run in parallel. Fact checking depends on their results, so it runs after.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.coordinator import sort_tasks_into_waves

## Correct Pattern: Parallel + Sequential Waves

In [ ]:
# Define tasks with data dependencies
tasks = [
    SubTask(task_id='web', agent_type='web_researcher', instruction='Search for remote work studies', context='Focus on 2024'),
    SubTask(task_id='data', agent_type='data_extractor', instruction='Query remote work statistics', context='remote_work_stats table'),
    SubTask(task_id='docs', agent_type='document_analyzer', instruction='Parse Stanford study', context='doc-remote-work-stanford'),
    SubTask(task_id='facts', agent_type='fact_checker', instruction='Verify claims', context='', depends_on=['web', 'data', 'docs']),
]

waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    ids = [t.task_id for t in wave]
    agents = [t.agent_type for t in wave]
    print(f'Wave {i}: {ids}')
    print(f'  Agents: {agents}')
    print(f'  Can run in parallel: {len(wave) > 1}')
    print()

In [ ]:
# Compare: all sequential (no parallelism)
sequential_tasks = [
    SubTask(task_id='web', agent_type='web_researcher', instruction='a', context=''),
    SubTask(task_id='data', agent_type='data_extractor', instruction='b', context='', depends_on=['web']),
    SubTask(task_id='docs', agent_type='document_analyzer', instruction='c', context='', depends_on=['data']),
    SubTask(task_id='facts', agent_type='fact_checker', instruction='d', context='', depends_on=['docs']),
]
seq_waves = sort_tasks_into_waves(sequential_tasks)
print(f'Parallel waves: {len(waves)} (wave 0 has {len(waves[0])} tasks)')
print(f'Sequential waves: {len(seq_waves)} (all single-task waves)')

## CCA Exam Tip

> Task decomposition questions ask whether two subtasks should run in parallel or sequentially.
> - If Subtask B needs output from Subtask A → sequential
> - If they work from independent inputs → parallel
> - 'Web search' and 'document parsing' are the canonical parallel pair
> - 'Fact checking' and 'report writing' are always sequential